In [21]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time
from pathlib import Path


In [22]:
video_path = Path("..") / "data" / "Video_Test_F2787_125743_01_VIDCKPT_sec.mpg"
cap = cv2.VideoCapture(video_path)

In [23]:
show_video = 0
while cap.isOpened():
    if not show_video: break
    ret, frame = cap.read()
 
    # if frame is read correctly ret is True
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # color = cv2.cvtColor(frame, cv2.COLOR_RGB2RGB)
 
    cv2.imshow('frame', frame)#, color)
    if cv2.waitKey(1) == ord('q'):
        break
cv2.destroyAllWindows()

### Mask creation

In [24]:
# define a mask to pick out the apparatus
cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # set first frame
_, first_frame = cap.read() 
frame_shape = first_frame.shape
mask = np.ones(frame_shape, dtype=np.uint8)
# mask[0:frame_shape[0]//2] = 0
# mask[:, 0:frame_shape[1]//2-25] = 0

# upper right triangle
upper_right = frame_shape[0]//2, frame_shape[1]
lower_right = frame_shape[0], frame_shape[1]
upper_left = frame_shape[0]//2, frame_shape[1]//3 * 2 - 35

# slope = (lower_right[0] - upper_left[0]) // (lower_right[1] - upper_left[1])

# intercept = -500

slope_right = 1
# intercept = upper_left[0] - slope * upper_left[1]
intercept_right = -425

slope_left_upper = -1
intercept_left_upper = 1200

slope_left_lower = 1
intercept_left_lower = -300

for x in range(frame_shape[1]):
    for y in range(frame_shape[0]):

        criteria = (
            y < slope_right*x+intercept_right, # below, since y goes from top to bottom
            y < slope_left_upper*x + intercept_left_upper,
            y > slope_left_lower*x + intercept_left_lower,
            x > frame_shape[1] - 100,
            y < frame_shape[0] // 2 + 115, 
            x < frame_shape[1]//2
        )
        
        for c in criteria:
            if c: mask[y, x, :] = 0

first_frame_masked = np.where(mask == 1, first_frame, 0)
cv2.imwrite(Path("..") /"figures" / "image.png", first_frame_masked)


True

### Singular frames 

In [25]:
target_frame = 1

def extract_im_and_hist(target_frame, mask, savefigs=True):
    cap.set(cv2.CAP_PROP_POS_FRAMES, target_frame)
    ret, frame = cap.read()
    if ret:
        # to show the video
        # cv2.imshow(f"Frame {target_frame}", frame)
        # cv2.waitKey(0)

        # to access pixel values
        y_top, y_bottom = frame.shape[0]//2, frame.shape[0] - 0
        x_left, x_right = frame.shape[1]//2, frame.shape[1] - 150


        # pixels_bgr = frame[y_top:y_bottom:, x_left:x_right]
        pixels_bgr = np.where(mask == 1, frame, 0)
        # print(pixels_bgr)

        # cv2.imshow("..", pixels_bgr)
        # cv2.waitKey(wait_time_ms)
        if savefigs:
            cv2.imwrite(Path("..") /"figures" / "image.png", pixels_bgr)

        # only do histogram on the unmasked pixels
        valid_pixels_blue = pixels_bgr[:, :, 0][mask[:, :, 0] == 1]
        valid_pixels_green = pixels_bgr[:, :, 1][mask[:, :, 0] == 1]
        valid_pixels_red = pixels_bgr[:, :, 2][mask[:, :, 0] == 1]

        bins = np.arange(0, 255, 10)
        counts_blue, bins_blue = np.histogram(valid_pixels_blue, bins)
        counts_green, bins_green = np.histogram(valid_pixels_green, bins)
        counts_red, bins_red = np.histogram(valid_pixels_red, bins)
        bins_blue = bins_blue[:-1]
        bins_green = bins_green[:-1]
        bins_red = bins_red[:-1]

        cv2.destroyAllWindows()
        if savefigs:
            fig, axs=plt.subplots(3, 1)
            axs[0].plot(bins_blue, counts_blue)
            axs[1].plot(bins_green, counts_green)
            axs[2].plot(bins_red, counts_red)
            axs[0].set_title("blue", c="blue")
            axs[1].set_title("green", c="green")
            axs[2].set_title("red", c="red")
            fig.tight_layout()

            fig.savefig(Path("..") / "figures" / "histogram.png")
            plt.close()
    return {
        "counts_blue": counts_blue,
        "counts_green": counts_green,
        "counts_red": counts_red,
    }
# target_frame = 10000
# extract_im_and_hist(target_frame)

In [26]:
target_frame = 10000 - 1000
for addition in range(0, 10000, 10):
    break
    extract_im_and_hist(target_frame+addition, mask)
    time.sleep(0.05)

In [27]:
target_frame = 0

for addition in range(0, 5000, 100):
    break
    extract_im_and_hist(target_frame+addition, mask=np.ones(frame_shape))
    time.sleep(0.1)


In [28]:
icing_frames = 10000-1000, 10000-100 + 10000
noicing_frames = 0, 5000

In [ ]:
counts = {
    "icing": {
        "counts_blue": [],
        "counts_green": [],
        "counts_red": []
    },
    "noicing": {
        "counts_blue": [],
        "counts_green": [],
        "counts_red": []
    },
}

for i in np.arange(icing_frames[0], icing_frames[1], 100):
    histogram_counts = extract_im_and_hist(i, mask, savefigs=False)
    for key in histogram_counts:
        counts["icing"][key].append(histogram_counts[key])
for i in np.arange(noicing_frames[0], noicing_frames[1], 100):
    extract_im_and_hist(i, mask, savefigs=0)
    histogram_counts = extract_im_and_hist(i, mask, savefigs=False)
    for key in histogram_counts:
        counts["icing"][key].append(histogram_counts[key])
